# Zero-shot Generation

## Overview

This notebook handles the first stage of the zero-shot pipeline. Its job is straightforward: take a set of prompts, run the model on them exactly as they are and save the raw outputs.

All prompts are collected from a single text file, keeping generation simple and making it easy to expand or modify the set of tested parameters without touching the notebook logic.

All generated samples are then evaluated in the separate notebook 02_zero_shot_evaluate.ipynb, where we check compilation validity, musical structure, adherence to constraints, and other metrics. Separating evaluation into its own notebook means we don’t have to regenerate outputs every time we update or improve our evaluation logic. The generations stay fixed and reproducible, while the evaluation pipeline can evolve independently.

## Objectives

- Provide a reproducible zero-shot generation workflow.

- Run the model on arbitrary prompts and store every response.

- Produce structured output files that will be consumed and evaluated later in 02_zero_shot_evaluate.ipynb.


## Notebook Structure

This notebook is organized into sections that:
1. Load and configure prompt to be tested

2. Initialize the OpenAI client for HuggingFace router models

3. Generate LilyPond code using the chosen model and parameters

4. Save all generations in a consistent directory structure

5. Record metadata for reproducibility and later evaluation

## Requirements

- **Python 3.8+**
- **HF_TOKEN**: HuggingFace API token (set as environment variable)
- **Model Access**: Access to the selected model via HuggingFace router

## Key Parameters

- **Model** – which model to query (e.g., GPT-OSS-20B)

- **Temperature** – randomness control (e.g., 0.7 for moderate variability)

- **Max Tokens** – upper limit of the generated sequence

- **Number of Runs** – how many samples to generate for each prompt


## Output

All raw generations are saved under:

`zero_shot_outputs/raw`

These raw outputs are then loaded, parsed, and evaluated in the next stage (02_zero_shot_evaluate.ipynb), which handles compilation checks, musical validity metrics, and summary tables.

In [ ]:
from openai import OpenAI
from IPython.display import display, Markdown
from pathlib import Path
from datetime import datetime
import os, re, json


# Helper: extract LilyPond code only
def extract_lilypond_code(text: str) -> str:
    """Extract LilyPond code from a model reply, removing markdown fences/preambles."""
    text = re.sub(r'```(?:lilypond)?\s*\n?(.*?)```', r'\1', text, flags=re.DOTALL)
    text = re.sub(r'^(?:Here|Here\'s|Here is|Output|LilyPond code:|The following).*?:\s*',
                  '', text, flags=re.IGNORECASE)
    return text.strip()


# Generator (saves in few_shot_outputs/raw/<subfolder>)
def run_generate(
    prompt_file: str | Path = "../configs/prompts/few_shot.txt",
    *,
    model: str = "openai/gpt-oss-20b:fireworks-ai",
    num_runs: int = 2,
    max_tokens: int = 4096,
    temperature: float = 0.7,
    output_folder_name: str = "session_01"
):

    prompt_path = Path(prompt_file)
    if not prompt_path.exists():
        display(Markdown(f"**❌ Error:** Prompt file not found: `{prompt_path}`"))
        return []

    hf_token = os.getenv("HF_TOKEN")
    if not hf_token:
        display(Markdown("**❌ Error:** `HF_TOKEN` environment variable not set!"))
        return []

    client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=hf_token)

    # Detect base directory safely (handles notebooks too)
    try:
        base_dir = Path(__file__).resolve().parent
    except NameError:
        base_dir = Path.cwd()  # fallback for Jupyter / REPL

    save_dir = base_dir / "few_shot_outputs" / "raw" / output_folder_name
    save_dir.mkdir(parents=True, exist_ok=True)

    prompt_text = prompt_path.read_text(encoding="utf-8").strip()
    display(Markdown(f"### 📝 Input Prompt\n```\n{prompt_text}\n```\n---"))

    all_results = []
    for i in range(1, num_runs + 1):
        display(Markdown(f"## 🔄 Run {i}/{num_runs}"))
        t0 = datetime.now()
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt_text}],
            max_tokens=max_tokens,
            temperature=temperature,
        )
        dt = (datetime.now() - t0).total_seconds()

        choice = resp.choices[0]
        raw_text = (choice.message.content or "") if choice.message else ""
        lp = extract_lilypond_code(raw_text)

        # Display output (no checks)
        md = f"### Model Output\n```lilypond\n{lp}\n```\n"
        md += f"⏱️ Time: {dt:.2f}s | Tokens: {resp.usage.total_tokens if resp.usage else '-'}"
        display(Markdown(md))
        display(Markdown("---"))

        # Save both raw and extracted
        (save_dir / f"raw_{i:02d}.txt").write_text(raw_text, encoding="utf-8")
        (save_dir / f"out_{i:02d}.ly").write_text(lp, encoding="utf-8")

        # Append metadata for summary.json
        all_results.append({
            "run": i,
            "elapsed_seconds": round(dt, 3),
            "tokens_used": resp.usage.total_tokens if resp.usage else None,
            "model": model,
            "temperature": temperature,
            "prompt_file": str(prompt_path)
        })

    # Write summary.json
    summary_path = save_dir / "summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    display(Markdown(f"✅ Saved all outputs and summary in `{save_dir}`"))
    return all_results

# Example:
# results = run_generate(
#     "../configs/prompts/few_shot.txt",
#     num_runs=3,
#     output_folder_name="test_batch_A"
# )

------------

Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Key</h3>
</div>

##  C Major

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key do \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="c_major" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key do \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  <<
    \new Staff {
      \key do \major
      \tempo 4 = 120
      r4 do'8 re'8 mi'8 fa'8 |
      sol'4 la'8 sol'8 fa'8 mi'8 |
      re'4 do'8 re'8 mi'8 fa'8 |
      sol'4 la'8 sol'8 fa'8 mi'8 |
      re'4 do'8 re'8 mi'8 dod'8 |
      sol'4 la'8 sol'8 fa'8 mi'8 |
      do'8 re'16 mi'16 fa'16 sol'16 la'16 si'16 do''16 re''16 |
      sol''4 re''8 do''8 r4 |
    }
  >>
}
```
⏱️ Time: 16.13s | Tokens: 3308

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    \key do \major
    \time 4/4
    \clef treble

    \tempo "Andante"

    do'8 re'8 mi'8 fa'8 | sol'8 la'8 do''8 re''8 |
    re''8 mi''8 fa''8 sol''8 | la''8 do'''8 re'''8 mi'''8 |

    do'8 mi'8 sol'8 mi'8 | re'8 do'8 la'8 sol'8 |
    do''8 mi''8 sol''8 mi''8 | re''8 do''8 la''8 sol''8 |

    \bar "|."
  }
}
```
⏱️ Time: 9.01s | Tokens: 2061

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\c_major`

-----------------

##  F Major

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key fa \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="f_major" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key fa \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    \key fa \major
    \time 4/4
    R4. R8
    la''8 si''8 sol''8 fa''8
    sol''8 la''8 do'''8 re'''8
    mi'''8 fa'''8 sol'''8 re'''8
    la''8 sol''8 fa''8 sol''8
    R4 r4

    re''8 re''8 re''8 re''8
    fad''8 re''8 dod''8 re''8
    re''8 re''8 re''8 re''8
    fad''8 re''8 dod''8 re''8
    R4 r4

    sol''8 la''8 do'''8 re'''8
    re'''8 sol'''8 fa'''8 sol'''8
    re''8 re''8 re''8 re''8
    fad''8 re''8 dod''8 re''8

    r2
  }
}
```
⏱️ Time: 22.64s | Tokens: 3007

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    \key fa \major

    R2.*2

    fa'4 la'4 mi'4 re'4
    sol'4 la'4 fa'4 re'4

    R2.*2

    fa'8 la'8 mi'8 re'8
    sol'8 la'8 fa'8 re'8

    R2.*3

    re'4 re'4 sol'4 sol'4
    la'4 la'4 mi'4 mi'4

    R2.*1

    fa'8 fa'8 sol'8 sol'8
    la'8 la'8 mi'8 mi'8

    R2.*4

    re'8 re'8 mi'8 mi'8
    fa'8 fa'8 sol'8 sol'8

    R2.*2

    r2 r2
  }
}
```
⏱️ Time: 6.64s | Tokens: 1855

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\f_major`

-----------

##   E Minor

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key mi \minor.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.


In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="e_minor" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key mi \minor.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    \key mi \minor
    \time 4/4

    R2.*4

    mi''8. re''16 mi''8. re''16 sol''4 ~ sol''8
    fa''8[ mi''16 re''16 mi''8. mi''16] la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16 mi''8[ mi''8 la''8. mi''16] fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R2.*3

    r4 r8 la''8 re''8 mi''16 fad''16 sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16] sol''16 la''16 fad''16 sol''16 mi''8 la''16 sol''16
    fad''4 r4 r2

    R2.*2

    r4 sol''4 ~ sol''8
  }
}
```
⏱️ Time: 18.91s | Tokens: 3089

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  <<
    \new Staff {
      \key mi \minor
      \time 4/4
      \clef treble
      % 8‑measure theme
      mi''8 re''8 mi''8 la''8 |            % 1
      re''8 mi''8 sol''4. sol''8 |          % 2
      fa''8 mi''8 re''8 la''8 |            % 3
      re''8 si''8 la''8 reb''8 |            % 4
      mi''8 re''8 mi''8 la''8 |            % 5
      re''8 mi''8 sol''4. sol''8 |          % 6
      fa''8 mi''8 re''8 la''8 |            % 7
      re''8 si''8 la''8 reb''8 |            % 8
    }
    \new Staff {
      \key mi \minor
      \time 4/4
      \clef bass
      % Simple accompaniment
      mi4 r4 | mi4 r4 | re4 r4 | reb4 r4 |
      mi4 r4 | mi4 r4 | re4 r4 | reb4 r4 |
    }
  >>
}
```
⏱️ Time: 25.00s | Tokens: 2910

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\e_minor`

-----------

##  G Major (1♯)

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key sol \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="g_major" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key sol \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  \new Staff {
    \key sol \major
    \time 4/4
    \clef treble

    sol'4 la'4 si'4 re''4
    re''8 mi''16 fa''16 sol''8 sol''8 la''8. sol''16
    sol''4 r4 r4 r4
    re''8 re''8 re''8 re''8 sol''8 la''8 re''8 sol''8

    la''4 si''4 la''4 sol''4
    la''8 si''8 la''8 sol''8 fa''8 sol''8 la''8 sol''8
    re''4 r4 r4 r4
    sol''8 sol''8 sol''8 sol''8 si''8 sol''8 fa''8 sol''8

    re''8 re''8 re''8 re''8 re''8 re''8 re''8 re''8
    sol''8 dod''8 sol''8 dod''8 sol''8 dod''8 sol''8 dod''8
    re''4 r4 r4 r4
    sol''2 r2
  }
}
```
⏱️ Time: 19.18s | Tokens: 3248

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  <<
    \new Staff \with { \clef treble } {
      \key sol \major
      R2.*4
      sol''4 la''8 sol''8 si''8 sol''8 la''8 sol''8
      r2
      re''8 re''8 re''8 re''8 si''8 re''8 si''8 re''8
      r2
      la''16 sol''16 fa''16 la''16 sol''16 la''16 sol''16 fa''16
      r2
      re''8 la''8 re''8 la''8 re''8 la''8 re''8 la''8
      r2
    }
  >>
}
```
⏱️ Time: 9.38s | Tokens: 2029

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\g_major`

----------------

##  D Major (2♯)

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key re \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="d_major" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key re \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    \key re \major
    \time 4/4

    % Intro
    R1*2
    re'4 mi'4 fa'4 sol'4
    la'4 si'8 re''8 la'8
    re''8 mi''8 fa''8 sol''8 la''8
    re''8 mi''8 fa''8 sol''8

    % Verse
    re'4 re'4 mi'4 mi'4
    fa'4 fa'4 sol'4 sol'4
    la'4 la'4 si'4 si'4
    re''4 re''4 mi''4 mi''4

    % Chorus
    re'8 re'8 mi'8 mi'8 fa'8 fa'8 sol'8 sol'8
    la'8 la'8 si'8 si'8 re''8 re''8 mi''8 mi''8

    \bar "|."
  }
}
```
⏱️ Time: 14.68s | Tokens: 2591

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    \key re \major
    R1
    re'8[ sol'8 la'8. sol'16 ] mi'8 re'8 fa'8 re'8
    re'4 re'4 re'4 re'4
    re''8 la''16 sol''16 mi''16 re''8 re''8
    re'8[ sol'8 la'8. sol'16 ] mi'8 re'8 fa'8 re'8
    R1
  }
}
```
⏱️ Time: 14.41s | Tokens: 2755

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\d_major`

----------------

##  B♭ Major (2♭)

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key sib \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.


In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="bes_major" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key sib \major.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  \key sib \major
  {
    R1*2
    sib''4 sib''8 re''8 reb''16 re''16 mi''8 mi''8 fa''8 fa''8
    sol''8 sol''8 re''8 re''8 re''8 re''8 mi''8 mi''8
    R2.*3
    sib''8 sib''8 re''8 reb''16 re''16 mi''8 mi''8 fa''8 fa''8
    sol''8 sol''8 re''8 re''8 re''8 re''8 mi''8 mi''8
    R1*3
  }
}
```
⏱️ Time: 23.54s | Tokens: 4002

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    \key sib \major
    R2*4
    sib'8 sib'8 reb'8 sib'8
    re'8 reb'8 sib'8 reb'8
    la'8 la'8 la'8 la'8
    sol'8 si'8 sol'8 si'8

    R2*2

    sib'4 la'8 la'8 sol'8 sol'8
    reb'8 re'8 sib'8 reb'8
    la'4 sol'8 la'8 sol'8 la'8

    r4 sib'8 sib'8 reb'8 sib'8
    re'8 reb'8 sib'8 reb'8
    la'8 la'8 la'8 la'8
    sol'8 si'8 sol'8 si'8

    \bar "|."
  }
}
```
⏱️ Time: 26.92s | Tokens: 4393

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\bes_major`

----------------

##  A Minor (0)

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key la \minor.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="a_minor" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key la \minor.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    \key la \minor
    \time 4/4

    R1*2

    la'8 la'8 la'8 la'8 si'8 la'8 sol'8 sol'8
    fa'8 fa'8 fa'8 fa'8 la'8 fa'8

    R2.*4

    re'8 re'8 re'8 re'8 fa'8 re'8
    dod'8 dod'8 dod'8 dod'8 fa'8 dod'8
    dod'!8 dod'8 dod'8 dod'8 fa'8 dod'8

    sol'4 sol'4 sol'2

    R2.*3

    la'8 la'8 la'8 la'8 si'8 la'8 sol'8 sol'8
    fa'8 fa'8 fa'8 fa'8 la'8 fa'8

    R2.

    re'8 re'8 re'8 re'8 fa'8 re'8
    dod'8 dod'8 dod'8 dod'8 fa'8 dod'8
  }
}
```
⏱️ Time: 6.74s | Tokens: 1679

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  \new Staff {
    \key la \minor
    \time 4/4

    % Intro
    R2.*2
    la''8 la''8 la''8 la''8
    re''8 mi''8 fa''8 sol''8
    la''4 si''8 la''8 sol''8 fa''8
    mi''8 re''8 la''8 la''8

    R2.*2

    % Verse
    la''4 la''4 la''4 la''4
    re''8 mi''8 fa''8 sol''8 la''8 si''8
    re''4 re''4 re''4 re''4
    la''8 la''8 la''8 la''8 re''8 mi''8

    R4 r4 r4 r4

    % Chorus
    la''4 la''4 la''4 la''4
    si''8 la''8 sol''8 fa''8 mi''8 re''8
    la''4 la''4 la''4 la''4
    re''8 mi''8 fa''8 sol''8 la''8 si''8

    \bar "|."
  }
}
```
⏱️ Time: 11.06s | Tokens: 2127

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\a_minor`

----------------

##  D Minor (1♭)

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key re \minor.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="d_minor" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \key re \minor.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  \new Staff {
    \clef treble
    \key re \minor
    \time 4/4
    \tempo 4 = 100

    % Intro
    R2.
    r4 re8. re16 si8 si8

    % First phrase
    re8 la16 re16 la16 re16 la8 la8
    re8 re8 re8 re8
    re4 r4 r8 re8

    % Second phrase
    re8 dod16 re16 mi16 la8 la8 re8 re8
    re4 r4 r8 re8

    % Repeated section
    \repeat volta 2 {
      re4 re8 re8 re8 re8
      re8 la16 re16 la16 re16 la8 la8
    }

    % Outro
    R2. *2
  }
}
```
⏱️ Time: 16.69s | Tokens: 2200

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    \key re \minor
    R4.*2
    re''8 re''8 re''8 re''8
    mi''16 re''16 la''8 re''8
    R2.
    re''16 fa''16 sol''16 la''16
    re''8 mi''8 si''8 re''8
    R4.*3
    re''4 re''4
    mi''8 la''8 re''8 mi''8
    R2.*2
    re''8 re''8 re''8 re''8
    re''8 re''8 re''8 re''8
    R4.*5
    re''4 re''4
    re''8 re''8 re''8
    r4
  }
}
```
⏱️ Time: 6.18s | Tokens: 1787

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\d_minor`

----------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Tempo</h3>
</div>

##   Slow Tempo

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \tempo 4 = 60.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="slow_tempo" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \tempo 4 = 60.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  \tempo 4 = 60
  {
    R1*2
    \repeat volta 2 {
      re''8 mi''16 fa''16 sol''8 la''8
      sol''8 fa''8 mi''8 re''8
      re''4 mi''4
    }
    \alternative {
      { re''8 mi''16 fa''16 sol''8 la''8 }
      { re''8 mi''16 fa''16 sol''8 la''8 }
    }
    \repeat volta 2 {
      la''8 sol''8 fa''8 mi''8
      re''8 mi''8 fa''8 sol''8
      re''4 mi''4
    }
    r2
  }
}
```
⏱️ Time: 9.90s | Tokens: 1884

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\tempo 4 = 60

\score {
  \new Staff {
    \clef treble
    \key do \major
    \time 4/4

    % Sezione A
    do''4 re''4 mi''4 fa''4 \bar "|"
    sol''4 la''4 si''4 do'''4 \bar "|"

    % Sezione B
    re'''8 mi'''8 fa'''8 sol'''8 la'''8 sol'''8 fa'''8 mi'''8 \bar "|"
    re'''4 do'''4 re'''4 do'''4 \bar "|"

    % Sezione C
    R4. \bar "|"

    % Sezione D (con note accidentate)
    dod''8 dod''8 re''8 re''8 mi''8 mi''8 fa''8 fa''8 \bar "|"
    R4. \bar "|"

    % Ripetizione della sezione A
    \repeat volta 2 {
      do''4 re''4 mi''4 fa''4 \bar "|"
      sol''4 la''4 si''4 do'''4 \bar "|"
    }
  }
}
```
⏱️ Time: 13.40s | Tokens: 2439

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\slow_tempo`

-----------

##   Fast Tempo

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \tempo 4 = 160.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="fast_tempo" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \tempo 4 = 160.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  \tempo 4 = 160
  {
    re''8 mi''16 fa''16 sol''8. la''8. re''16
    sol''8 la''16 si''16 re''8. mi''8. re''16
    re''4 r8 re''8 mi''8
    fa''8 sol''8 la''8. si''8. re''16

    R2.*2

    re''8 mi''16 fa''16 sol''8. la''8. re''16
    sol''8 la''16 si''16 re''8. mi''8. re''16
    re''4 r8 re''8 mi''8
    fa''8 sol''8 la''8. si''8. re''16

    R2.*2

    re''8 mi''16 fa''16 sol''8. la''8. re''16
    sol''8 la''16 si''16 re''8. mi''8. re''16
    re''4 r8 re''8 mi''8
    fa''8 sol''8 la''8. si''8. re''16

    R2.
  }
}
```
⏱️ Time: 12.81s | Tokens: 2494

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\tempo 4 = 160

\score {
  \new Staff {
    \clef treble
    \key do \major
    \time 4/4

    % Intro
    R1*2
    do'8 re' mi' fa' | sol' la' si' dod' |
    do'' re'' mi'' fa'' | sol'' la'' si'' dod''

    % Theme
    \repeat volta 2 {
      do''8 re'' mi'' fa'' | sol'' la'' si'' dod'' |
      do'8 re' mi' fa' | sol' la' si' dod' |
      re''8 mi'' fa'' sol'' | la'' si'' dod'' re'' |
      re'8 mi' fa' sol' | la' si' dod' re' |
    }

    % Bridge
    \tempo 4 = 120
    \repeat volta 2 {
      sol''8 la'' si'' dod'' | la'' sol'' re'' mi'' |
      re''8 si'' la'' sol'' | la'' sol'' re'' mi'' |
    }

    % Finale
    do''8 re'' mi'' fa'' | sol'' la'' si'' dod'' |
    do''8 re'' mi'' fa'' | sol'' la'' si'' dod'' |
    R1
  }
}
```
⏱️ Time: 6.99s | Tokens: 1836

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\fast_tempo`

-----------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Time Signature</h3>
</div>

##  3/4 (simple triple)

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \time 3/4.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="three_four" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \time 3/4.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  \new Staff {
    \time 3/4
    r2. \break
    do'4 re' mi' \break
    sol'4 la' si' \break
    re''4 mi'' fa'' \break
    sol''4 la'' si'' \break
    re''4 re'' re'' \break
    la''4 la'' la'' \break
    do''4 re'' mi'' \break
    sol''4 la'' si'' \break
    re''4 mi'' fa'' \break
    sol''4 la'' re'' \break
    fad'8 re'8 mi'4 \break
    re''8 re''8 re''4 \bar "|."
  }
}
```
⏱️ Time: 13.71s | Tokens: 2596

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  <<
    \new Staff <<
      \time 3/4
      \tempo "Allegro moderato" 4=120
      \repeat volta 2 {
        r4 do'8 re'8 mi'8 |
        fa'4 sol'8 la'8 si'8 |
        re'4 mi'8 fa'8 sol'8 |
        la'4 si'8 do''8 re''8 |
        mi''4 re''8 la'8 sol'8 |
        fa'4 mi'8 re'8 do'8 |
        si'4 la'8 sol'8 fa'8 |
        sol'4 fa'8 re'8 mi'8 |
      }
      \alternative {
        { r4 r8 r8 r8 | \bar "|." } 
        { r4 do''8 re''8 mi''8 | \bar "|." } 
      }
    >>
  >>
}
```
⏱️ Time: 18.67s | Tokens: 3377

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\three_four`

-------------------

##  6/8 (compound duple)

prompt:
Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \time 6/8.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="six_eight" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond musically as good as the examples you see.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Use \time 6/8.
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  \new Staff {
    \clef treble
    \time 6/8
    \key do \major
    \repeat volta 2 {
      r4. do''8 re''8 mi''8
      sol''8 la''8 si''8 re''16 sol''16 mi''8
      re''8 mi''8 fa''8 sol''8 la''8 do''8
      do''4. la''8 sol''8 fa''8
      mi''8 re''8 do''8 la''16 sol''16 fa''8
      sol''8 si''8 la''8 sol''8 fa''8 mi''8
      re''8 re''8 re''8 re''8 re''8 re''8
      sol''8 mi''8 fa''8 sol''8 la''8 do''8
      re''8 do''8 la''8 sol''8 fa''8 mi''8
      sol''8 si''8 la''8 sol''8 fa''8 mi''8
      re''8 re''8 re''8 re''8 re''8 re''8
      r4. do''8 re''8 mi''8
    }
    \bar "|."
  }
}
```
⏱️ Time: 10.16s | Tokens: 2332

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond

```
⏱️ Time: 49.86s | Tokens: 5055

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\six_eight`

-----------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Harmony </h3>
</div>

##  Dyads (2-note chords)

prompt:
Write a LilyPond passage consisting only of dyads (2-note chords), not single notes.
Every event must be a dyad written using LilyPond chord brackets < >.
Each dyad must contain exactly two notes with one shared duration.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="dyads" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond passage consisting only of dyads (2-note chords), not single notes.
Every event must be a dyad written using LilyPond chord brackets < >.
Each dyad must contain exactly two notes with one shared duration.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    <do' re'>4 <mi' sol'>8 <fa' la'>8 <si' do''>4
    <re'' mi''>2
    <re' mi'>8 <fa' sol'>8 <la' si'>8 <do'' re''>8
    <mi'' fa''>4 <sol'' la''>4
    <do'' re''>4 <mi'' sol''>8 <fa'' la''>8 <si' do''>4
    <re' mi'>2
  }
}
```
⏱️ Time: 9.69s | Tokens: 1563

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    <do' re'>4 <mi' fa'>4 <sol' la'>4 <si' do''>4
    <do' mi'>8 <re' sol'>8 <fa' la'>8 <si' do''>8
    <do' re'>2 <mi' fa'>2
    <sol' la'>1
    <re' mi'>4 <do' sol'>4 <fa' re'>4 <mi' si'>4
    <re' do'>8 <mi' fa'>8 <sol' la'>8 <si' do''>8
  }
}
```
⏱️ Time: 17.58s | Tokens: 3230

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\dyads`

------------

##  Triads (3-note chords)

prompt:
Write a LilyPond passage consisting only of triads (3-note chords), not single notes.
Every event must be a triad written using LilyPond chord brackets < >.
Each triad must contain exactly three notes with one shared duration.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.

In [ ]:
results = run_generate(     "../../configs/prompts/few_shot.txt",     num_runs=5,     output_folder_name="triads" )

### 📝 Input Prompt
```
Example1:
\version "2.24.4"
\language "italiano"

\score {
  {
    R1
    r4 fad''16 mi''16 fad''16 sol''16
    la''8[ si''8 la''8. sol''16 ]
    fad''8 re''8
    sol''4 ~ sol''8[ sol''8 fa''8. fa''16 ]
    fa''8[ mi''16 re''16 mi''8. mi''16 ]
    la''16 sol''16 fa''16 mi''16
    re''8 sol''16 fa''16
    mi''8[ mi''8 la''8. mi''16 ]
    fad''!8 sol''16 la''16 fad''8. sol''16
    sol''4 r4 r2

    R1*3

    r4 r8 la''8
    re''8 mi''16 fad''16
    sol''8 fad''16 sol''16
    mi''8[ mi''8 la''8. la''16 ]
    sol''16 la''16 fad''16 sol''16
    mi''8 la''16 sol''16
    fad''4 r4 r2

    R1*2

    r4 sol''4 ~ sol''8
  }
}

Example2:
\version "2.24.4"
\language "italiano"

\score {
  {
    re'4 fad'4 la'4 re'4

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''8[ dod''16 si'16 la'16 sol'16 fad'16 mi'16 ]
    re'16 mi'16 fad'16 re'16 la'16 si'16 dod''16 la'16

    re''4 re'4

    la''16 si''16 la''16 fad''16 la''16 si''16 la''16 fad''16
    sol''16 la''16 sol''16 mi''16 sol''16 la''16 sol''16 mi''16
    fad''16 sol''16 fad''16 re''16 fad''16 sol''16 fad''16 re''16
    mi''16 fad''16 mi''16 dod''16 mi''16 fad''16
  }
}

Example3:
\version "2.24.4"
\language "italiano"

\score {
  {
    R2.*11

    sol''8 sol''8 sol''8 sol''8 si''8 sol''8
    fad''8 fad''8 fad''8 fad''8 si''8 fad''8

    R2.*13

    re''8 re''8 re''8 re''8 fad''8 re''8
    dod''8 dod''8 dod''8 dod''8 fad''8 dod''8

    dod''!8 dod''8 dod''8 dod''8 fad''8 dod''8

    si'8 si'8 si'8 si'8 mi''8 si'8
    si'8 si'8 si'8 si'8 mi''8 si'8
    lad'8 dod''8 dod''8 dod''8 mi''8 dod''8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8

    re''8 si'8 si'8 si'8 fad''8 re''8
    dod''8 lad'8 lad'8 lad'8
  }
}

Write a LilyPond passage consisting only of triads (3-note chords), not single notes.
Every event must be a triad written using LilyPond chord brackets < >.
Each triad must contain exactly three notes with one shared duration.
Always use \language "italiano" and Italian note names (do re mi fa sol la si, dod, reb, lad…).
Output LilyPond code only.
Do not use \relative anywhere.
Put \version "2.24.4" on the first line.
Pay attention to examples but do not copy them.
```
---

## 🔄 Run 1/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    <do mi sol>4 <re fa la>4 <mi sol si>4 <fa la do>4 <sol si re>4 <la do mi>4 <si re fa>4 <do mi sol>4
  }
}
```
⏱️ Time: 6.83s | Tokens: 1827

---

## 🔄 Run 2/5

...

---

## 🔄 Run 5/5

### Model Output
```lilypond
\version "2.24.4"
\language "italiano"

\score {
  {
    <do re mi>4 <re mi fa>4 <mi fa sol>4 <fa sol la>4
    <sol la si>4 <la si do'>4 <si do' re'>4 <do' re' mi'>4
    <re' mi' fa'>4 <mi' fa' sol'>4 <fa' sol' la'>4 <sol' la' si'>4
    <la' si' do''>4 <si' do'' re''>4 <do'' re'' mi''>4 <re'' mi'' fa''>4
  }
}
```
⏱️ Time: 10.62s | Tokens: 2076

---

✅ Saved all outputs and summary in `d:\uniPadova\Thesis\lilynorm\notebooks\few_shot\zero_shot_outputs\raw\triads`